In [ ]:
import torch
import torch.nn as nn
# from torchvision.models.video import s3d, S3D_Weights
import json
import cv2
import numpy as np
import random
from tqdm.auto import tqdm
import time
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from transformers import VideoMAEModel, VideoMAEConfig
from peft import LoraConfig, get_peft_model

%load_ext autoreload 
%autoreload 2


cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from utils.dataset import WLASLDataset

In [13]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [ ]:
class VideoMAEFinetune(nn.Module):
    def __init__(self, num_classes=300, mode='FFT', r=16, lora_alpha=32, lora_dropout=0.2):
        super().__init__()
        self.backbone = VideoMAEModel.from_pretrained(
            "CHANGE TO YOUR PATH",
            local_files_only=True
        )
        # 全部解冻
        if mode == 'FFT':
            for param in self.backbone.parameters():
                param.requires_grad = True
        elif mode == 'lora':
            lora_config = LoraConfig(
                r=r,                            
                lora_alpha=lora_alpha,        
                target_modules=["query", "value"],  
                lora_dropout=lora_dropout,
                bias="none"
            )
            self.backbone = get_peft_model(self.backbone, lora_config)

        hidden_size = self.backbone.config.hidden_size  # 768
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(p=0.3),    
            nn.Linear(hidden_size, num_classes)
        )

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"[{mode}] Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    def forward(self, x):
        x = x.permute(0, 2, 1, 3, 4)   # [B,C,T,H,W] → [B,T,C,H,W]
        outputs = self.backbone(pixel_values=x)
        features = outputs.last_hidden_state.mean(dim=1)  # [B, 768]
        return self.classifier(features)

In [15]:
def build_optimizer(model):
    
    backbone_trainable = [p for p in model.backbone.parameters() if p.requires_grad]
    classifier_trainable = [p for p in model.classifier.parameters() if p.requires_grad]
    
    return optim.AdamW([
        {'params': backbone_trainable,'lr': 5e-5},
        {'params': classifier_trainable,'lr': 1e-3},
    ], weight_decay=0.05)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 45
LEARNING_RATE = 1e-3
WORKERS = 2
NETWORK = "vit"
NUM_FRAMES = 16 # 32 for S3D
NUM_CLASSES = 300
JSON_FILE = ""
VIDEO_ROOT = ""
MODE = "lora"
CHECKPOINT_PATH = ""

In [17]:
def run_epoch(model, loader, optimizer, criterion, scaler, device, epoch, EPOCHS, train=True, network="cnn"):

    avg_lat, fps = 0.0, 0.0 
    total_time = 0.0
    latencies = []

    model.train() if train else model.eval() 

    total_loss, correct, total = 0.0, 0, 0
    d = "Train" if train else "Val"
    pbar = tqdm(loader, desc=d+f" [{epoch+1}/{EPOCHS}]", leave=False)

    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        t0 = time.perf_counter()
        
        if train:
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(inputs).view(inputs.size(0), -1)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                with autocast('cuda'):
                    outputs = model(inputs).view(inputs.size(0), -1)
                    loss = criterion(outputs, labels)

                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                t1 = time.perf_counter()

                batch_time = t1 - t0
                total_time += batch_time
                latencies.append(batch_time / inputs.size(0) * 1000)  # ms/video

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.3f}",
                         acc=f"{100.*correct/total:.1f}%")

    avg_loss = total_loss / len(loader)
    acc = 100. * correct / total

    if not train and latencies:
        avg_lat = np.mean(latencies)
        fps = total / total_time
        print(f"Val Acc: {acc:.2f} | Latency: {avg_lat:.2f} ms/video | FPS: {fps:.1f}")

    return avg_loss, acc, avg_lat, fps

In [18]:
if __name__ == "__main__":
    # ── 数据集 ──
    train_set = WLASLDataset(JSON_FILE, VIDEO_ROOT, split='train',
                             num_frames=NUM_FRAMES)
    val_set = WLASLDataset(JSON_FILE, VIDEO_ROOT, split='val',
                             num_frames=NUM_FRAMES,
                             label_map=train_set.action_to_idx)
    test_set = WLASLDataset(JSON_FILE, VIDEO_ROOT, split='test',
                             num_frames=NUM_FRAMES,
                             label_map=train_set.action_to_idx)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=WORKERS, pin_memory=True,
                              persistent_workers=True  )
    val_loader = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=WORKERS, pin_memory=True,
                              persistent_workers=True  )
    test_loader = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=WORKERS, pin_memory=True,
                              persistent_workers=True  )

    print(DEVICE)

    # print("Scanning for corrupted videos...")
    # corrupted = []
    # for vid_id in train_set.video_ids + val_set.video_ids + test_set.video_ids:
    #     path = os.path.join(VIDEO_ROOT, f"{vid_id}.mp4")
    #     cap  = cv2.VideoCapture(path)
    #     ret, _ = cap.read()
    #     if not ret:
    #         corrupted.append(vid_id)
    #     cap.release()
    # print(f"Corrupted videos: {len(corrupted)} / "
    #       f"{len(train_set)+len(val_set)+len(test_set)} total")
    
    model = VideoMAEFinetune(NUM_CLASSES, mode=MODE)
    model.to(DEVICE)

    optimizer = build_optimizer(model)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = GradScaler('cuda')

    # checkpoint = torch.load(CHECKPOINT_PATH)
    # model.load_state_dict(checkpoint['model_state_dict'])

    best_val_acc = 0.0

    for epoch in range(EPOCHS):
        t0 = time.time()

        train_loss, train_acc, _, _ = run_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE, epoch, EPOCHS, train=True, network=NETWORK)
        val_loss, val_acc, val_lat, val_fps = run_epoch(model, val_loader, optimizer, criterion, scaler, DEVICE, epoch, EPOCHS, train=False, network=NETWORK)

        scheduler.step()

        duration = time.time() - t0
        
        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
              f"Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | "
              f"Val Loss {val_loss:.4f} Acc {val_acc:.2f}% | "
              f"{duration:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),  
                'val_acc': val_acc,
                'label_map': train_set.action_to_idx,
            }, CHECKPOINT_PATH)
            print(f"✅ Best model saved (val acc {val_acc:.2f}%)")

    print("\n" + "="*50)
    ckpt = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(ckpt['model_state_dict'])
    _, test_acc, test_lat, test_fps = run_epoch(
        model, test_loader, None, criterion, scaler, DEVICE, 0, 1, train=False)
    print(f"Final Test Acc: {test_acc:.2f}% | Test Latency: {test_lat:.2f} | Test FPS: {test_fps:.1f}")

[TRAIN] 1897 videos | 300 classes
[VAL] 446 videos | 300 classes
[TEST] 317 videos | 300 classes
cuda


Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: /home/haod6/assignment3/model/vit/ssv2
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
fc_norm.weight    | UNEXPECTED |  | 
fc_norm.bias      | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[lora] Trainable: 822,060 / 87,047,724 (0.94%)


Train [1/45]:   0%|          | 0/30 [00:00<?, ?it/s]

...

[h264 @ 0x5c37c484ee40] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x5c37c484ee40] missing picture in access unit with size 10780
[h264 @ 0x5c37c4293ac0] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x5c37c4293ac0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5c37c5ecef80] stream 1, offset 0x2a27a7: partial file
[h264 @ 0x5c37c48c4a00] Invalid NAL unit size (745 > 472).
[h264 @ 0x5c37c48c4a00] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5c37c89ec4c0] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5c37c89ec4c0] stream 1, offset 0x3b7d3: partial file


Val [45/45]:   0%|          | 0/7 [00:00<?, ?it/s]

Val Acc: 19.73 | Latency: 2.50 ms/video | FPS: 399.0
Epoch [45/45] Train Loss 2.0274 Acc 78.23% | Val Loss 4.2415 Acc 19.73% | 77.8s



Val [1/1]:   0%|          | 0/5 [00:00<?, ?it/s]

Val Acc: 16.72 | Latency: 3.04 ms/video | FPS: 327.5
Final Test Acc: 16.72% | Test Latency: 3.04 | Test FPS: 327.5
